In [1]:
"""
RSA 공개키 암호 실습 프로그램

Rivest, Shamir, Adleman (1977)이 제안한 RSA 알고리즘을 구현합니다.
- 공개키 (n, e): 암호화에 사용, 누구나 알 수 있음
- 개인키 (n, d): 복호화에 사용, 소유자만 보관
- 보안 기반: 소인수분해 문제 (n = p*q에서 p, q를 구하기 어려움)

RSA_공개키암호_설명자료.md와 함께 사용하세요.
"""

import random
from typing import Tuple


# ---------------------------------------------------------------------------
# 수학적 보조 함수
# ---------------------------------------------------------------------------

def is_prime(n: int) -> bool:
    """n이 소수인지 간단히 판별 (작은 n용)."""
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for d in range(3, int(n ** 0.5) + 1, 2):
        if n % d == 0:
            return False
    return True


def gcd(a: int, b: int) -> int:
    """유클리드 알고리즘으로 최대공약수 계산."""
    while b:
        a, b = b, a % b
    return a


def extended_gcd(a: int, b: int) -> Tuple[int, int, int]:
    """
    확장 유클리드 알고리즘.
    Returns: (g, x, y) such that a*x + b*y = g = gcd(a, b)
    """
    if a == 0:
        return b, 0, 1
    g, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return g, x, y


def mod_inverse(a: int, m: int) -> int:
    """
    a의 모듈로 m에 대한 역원을 구합니다.
    a * x ≡ 1 (mod m) 인 x를 반환.
    """
    g, x, _ = extended_gcd(a % m, m)
    if g != 1:
        raise ValueError(f"역원이 존재하지 않음: gcd({a}, {m}) = {g}")
    return (x % m + m) % m


def generate_prime(bits: int = 8) -> int:
    """
    bits 비트 크기의 소수를 생성합니다.
    실습용으로 8~16비트 정도 사용 (실무에서는 1024비트 이상).
    """
    low = 2 ** (bits - 1)
    high = 2 ** bits - 1
    while True:
        p = random.randrange(low, high + 1)
        if p % 2 == 0:
            p += 1
        if is_prime(p):
            return p


# ---------------------------------------------------------------------------
# RSA 키 생성, 암호화, 복호화
# ---------------------------------------------------------------------------

def rsa_keygen(p: int = None, q: int = None, bits: int = 8) -> Tuple[Tuple[int, int], Tuple[int, int, int]]:
    """
    RSA 키 쌍을 생성합니다.
    
    Args:
        p, q: 소수 (None이면 자동 생성)
        bits: p, q 생성 시 비트 수 (p, q가 주어지면 무시)
    
    Returns:
        (public_key, private_key)
        public_key = (n, e)
        private_key = (n, d) 또는 (p, q, d)
    """
    if p is None or q is None:
        p = generate_prime(bits)
        q = generate_prime(bits)
        while p == q:
            q = generate_prime(bits)
    
    n = p * q
    phi = (p - 1) * (q - 1)
    
    # 공개 지수 e: gcd(e, phi) = 1. 보통 65537, 실습용으로 3~65537 사이에서 선택
    e = 65537
    if e >= phi or gcd(e, phi) != 1:
        for e in range(3, phi, 2):
            if gcd(e, phi) == 1:
                break
    
    d = mod_inverse(e, phi)
    
    public_key = (n, e)
    private_key = (n, d)  # (p, q, d) 형태로 저장해도 되나, 여기서는 (n, d)로 통일
    
    return public_key, private_key


def rsa_encrypt(M: int, public_key: Tuple[int, int]) -> int:
    """
    RSA 암호화: C = M^e mod n
    """
    n, e = public_key
    if M < 0 or M >= n:
        raise ValueError("평문 M은 0 <= M < n 이어야 합니다.")
    return pow(M, e, n)


def rsa_decrypt(C: int, private_key: Tuple[int, int]) -> int:
    """
    RSA 복호화: M = C^d mod n
    """
    n, d = private_key
    if C < 0 or C >= n:
        raise ValueError("암호문 C는 0 <= C < n 이어야 합니다.")
    return pow(C, d, n)


# ---------------------------------------------------------------------------
# 문자열 메시지 암호화 (블록 단위)
# ---------------------------------------------------------------------------

def message_to_blocks(msg: str, block_bits: int = 8) -> list:
    """
    문자열을 정수 블록 리스트로 변환.
    각 문자를 ASCII로, block_bits 비트 단위로 묶음.
    실습용: block_bits=8 (한 문자씩)
    """
    blocks = []
    for ch in msg:
        blocks.append(ord(ch))
    return blocks


def blocks_to_message(blocks: list) -> str:
    """정수 블록 리스트를 문자열로 복원."""
    return "".join(chr(b) for b in blocks)


def rsa_encrypt_message(msg: str, public_key: Tuple[int, int]) -> list:
    """
    문자열 메시지를 RSA로 암호화.
    n이 작으면 한 문자씩만 암호화 가능. (실습용)
    """
    n, _ = public_key
    blocks = message_to_blocks(msg)
    encrypted = []
    for m in blocks:
        if m >= n:
            raise ValueError(f"n={n}이 너무 작아 문자(ASCII {m})를 암호화할 수 없습니다.")
        encrypted.append(rsa_encrypt(m, public_key))
    return encrypted


def rsa_decrypt_message(encrypted: list, private_key: Tuple[int, int]) -> str:
    """암호문 블록 리스트를 복호화하여 문자열로 복원."""
    blocks = [rsa_decrypt(c, private_key) for c in encrypted]
    return blocks_to_message(blocks)


# ---------------------------------------------------------------------------
# 시연
# ---------------------------------------------------------------------------

def run_demo_numeric() -> None:
    """숫자 메시지에 대한 RSA 암·복호화 시연."""
    print("[1] 숫자 메시지 암·복호화 시연")
    print("-" * 50)
    
    # 실습용: 작은 소수 사용 (실제로는 수백~수천 비트)
    p, q = 61, 53  # 예시로 잘 알려진 소수
    print(f"소수 p = {p}, q = {q}")
    
    public_key, private_key = rsa_keygen(p=p, q=q)
    n, e = public_key
    _, d = private_key
    
    print(f"n = p × q = {n}")
    print(f"φ(n) = (p-1)(q-1) = {(p-1)*(q-1)}")
    print(f"공개키 (n, e) = ({n}, {e})")
    print(f"개인키 (n, d) = ({n}, {d})")
    print()
    
    M = 123  # 평문
    C = rsa_encrypt(M, public_key)
    M2 = rsa_decrypt(C, private_key)
    
    print(f"평문 M = {M}")
    print(f"암호화: C = M^e mod n = {C}")
    print(f"복호화: M' = C^d mod n = {M2}")
    print(f"복원 성공: {M == M2}")
    print()


def run_demo_text() -> None:
    """문자열 메시지에 대한 RSA 암·복호화 시연."""
    print("[2] 문자열 메시지 암·복호화 시연")
    print("-" * 50)
    
    # n이 256 이상이어야 ASCII 문자(0~255) 암호화 가능
    # p=61, q=53 -> n=3233 (실습용 고전 예시)
    public_key, private_key = rsa_keygen(p=61, q=53)
    n, e = public_key
    
    msg = "RSA"
    print(f"평문: \"{msg}\"")
    print(f"공개키 n = {n} (각 문자 ASCII 값 < n 이어야 함)")
    print()
    
    encrypted = rsa_encrypt_message(msg, public_key)
    print(f"암호문 (블록): {encrypted}")
    
    decrypted = rsa_decrypt_message(encrypted, private_key)
    print(f"복호화: \"{decrypted}\"")
    print(f"복원 성공: {msg == decrypted}")
    print()


def run_demo_random_keys() -> None:
    """랜덤 키 생성 및 암·복호화 시연."""
    print("[3] 랜덤 키 생성 시연")
    print("-" * 50)
    
    print("8비트 소수로 키 생성 (실습용, 실무에서는 1024비트 이상)")
    public_key, private_key = rsa_keygen(bits=8)
    n, e = public_key
    _, d = private_key
    
    print(f"공개키 (n, e): n은 {n.bit_length()}비트, e = {e}")
    print()
    
    # n보다 작은 평문만 가능
    M = random.randrange(2, min(n, 256))
    C = rsa_encrypt(M, public_key)
    M2 = rsa_decrypt(C, private_key)
    
    print(f"평문 M = {M}")
    print(f"암호문 C = {C}")
    print(f"복호화 M' = {M2}, 일치: {M == M2}")
    print()


# ---------------------------------------------------------------------------
# 메인
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    print("=" * 60)
    print("RSA 공개키 암호 실습")
    print("=" * 60)
    print()
    
    run_demo_numeric()
    run_demo_text()
    run_demo_random_keys()
    
    print("=" * 60)
    print("실습 완료. RSA_공개키암호_설명자료.md를 참고하세요.")
    print("=" * 60)


RSA 공개키 암호 실습

[1] 숫자 메시지 암·복호화 시연
--------------------------------------------------
소수 p = 61, q = 53
n = p × q = 3233
φ(n) = (p-1)(q-1) = 3120
공개키 (n, e) = (3233, 7)
개인키 (n, d) = (3233, 1783)

평문 M = 123
암호화: C = M^e mod n = 2868
복호화: M' = C^d mod n = 123
복원 성공: True

[2] 문자열 메시지 암·복호화 시연
--------------------------------------------------
평문: "RSA"
공개키 n = 3233 (각 문자 ASCII 값 < n 이어야 함)

암호문 (블록): [1077, 1825, 1317]
복호화: "RSA"
복원 성공: True

[3] 랜덤 키 생성 시연
--------------------------------------------------
8비트 소수로 키 생성 (실습용, 실무에서는 1024비트 이상)
공개키 (n, e): n은 16비트, e = 5

평문 M = 152
암호문 C = 7980
복호화 M' = 152, 일치: True

실습 완료. RSA_공개키암호_설명자료.md를 참고하세요.
